# 8. Собственные моды: независимая проверка и дальний сигнал

Строим дополнительный стационарный решатель. Он суммирует затухающие
пространственные моды и не использует осциллирующую квадратуру по $k$.
Это исследовательская ветка: усечение здесь иное, чем точный свободный хвост
основного решателя, и сходимость проверяется отдельно.

In [ ]:
from pathlib import Path
import sys, time, platform
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Notebook can be started from the repo root or notebooks/course.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/lighthit').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook inside the LightHit repository')
sys.path.insert(0, str(ROOT / 'src'))
from lighthit import Medium, SolverSettings, PointGreenSolver

# Explicit public test medium. No private provider is imported.
medium = Medium(0.04, 0.05, 0.7, 1.35, 450.0, 'course-synthetic')
np.set_printoptions(precision=7, suppress=True)
print('Python:', sys.executable)
print('Platform:', platform.platform())

## 8.1. Симметричная задача на собственные значения

При $\omega=0$ положим $d_\ell=\mu_t-\gamma_\ell>0$,
$\mathsf A=\operatorname{diag}(d_\ell)$,
$C_{\ell,\ell+1}=C_{\ell+1,\ell}=a_{\ell+1}$.
Конечное $P_N$-уравнение: $(\mathsf A+ikC)h=\sqrt2e_0$.
Преобразуем его матрицей $\mathsf A^{-1/2}$:
$$W=\mathsf A^{-1/2}C\mathsf A^{-1/2},\qquad
Wv_n=\lambda_n v_n.$$
$W$ вещественно-симметрична; её ненулевые собственные значения образуют
пары $\pm\lambda_n$. Для изотропного источника
$$\widetilde Q(k)=\frac1{\mu_a}\sum_n\frac{|v_{0n}|^2}{1+ik\lambda_n}.$$
Объединяя пары и обращая $1/(1+k^2\lambda_n^2)$ аналитически, получаем при $r>0$
$$\boxed{Q_N(r)=\sum_{\lambda_n>0}
\frac{|v_{0n}|^2}{2\pi\mu_a r\lambda_n^2}e^{-r/\lambda_n}.}$$
Все слагаемые положительны. Нулевая собственная величина при нечётном
размере матрицы даёт только распределение в начале координат, которое
исключено из этой задачи.

## 8.2. Радиальный ток и контроль нормировки

Вне источника $\nabla\cdot\mathbf F+\mu_aQ=0$. Для каждой моды
$$F_n(r)=\mu_a\lambda_n\left(1+\frac{\lambda_n}{r}\right)Q_n(r).$$
Отсюда $F/Q$ — средний радиальный косинус. Этот вывод использует закон
сохранения и служит отдельной проверкой нормировки.

Применяем только открытые параметры $\mu_a=0.07$, $\mu_s=0.022$, $g=0.9$.

In [ ]:
from lighthit.experimental.stationary_modes import StationaryModes,collision_components
water=Medium(.07,.022,.9,1.36,450.,'synthetic')
r=np.array([11.,20.,38.,69.,128.,225.,400.,800.])
models=[StationaryModes.build(water,N) for N in [64,128,256]]
q,f=collision_components(r,models[-1])
print('r, total charge, mean cosine one, mean cosine >=2, mean cosine total')
for row in np.column_stack([r,q.sum(1),f[:,1]/q[:,1],f[:,2]/q[:,2],f.sum(1)/q.sum(1)]):
    print(f'{row[0]:6.1f} {row[1]:.8e} {row[2]:.7f} {row[3]:.7f} {row[4]:.7f}')
for m in models:
    v,_=m.scalar_current(r)
    print('N =',m.degree,'relative change vs N=256:',np.max(np.abs(v-q.sum(1))/q.sum(1)))
fig,ax=plt.subplots();ax.plot(r,f[:,1]/q[:,1],'o-',label='one');ax.plot(r,f[:,2]/q[:,2],'o-',label='>=2');ax.plot(r,f.sum(1)/q.sum(1),'o-',label='all');ax.set(xlabel='r [m]',ylabel='mean radial cosine',xscale='log');ax.legend();plt.show()

## 8.3. Что видно в результате

Многократно рассеянный свет остаётся направленным. Однако его средний косинус
сначала растёт, затем уменьшается. Средний косинус **всего** света в данном
примере убывает. Поэтому название «всё сильнее коллимируется с расстоянием»
не описывает полученную зависимость.

При достаточно большом $r$ доминирует наибольшее положительное $\lambda_*$,
если она отделена от непрерывного спектра. $\kappa_*=1/\lambda_*$.
Ведущая плоская угловая мода удовлетворяет
$$(\mu_t-V)\psi=\kappa_*\mu\psi.$$
Интеграл по углу даёт $\langle\mu\rangle_\infty=\mu_a/\kappa_*$.
Для сферического источника ведущая поправка к току содержит $1/(\kappa_*r)$.
Доминирование одной моды нужно измерить, не предполагать на любом расстоянии.

In [ ]:
m=models[-1];qp,fp=m.scalar_current(r,leading_only=True)
print('kappa [m^-1]:',m.leading_attenuation_per_m)
print('limiting cosine:',m.leading_mean_cosine)
print('r, leading-mode fraction:',np.column_stack([r,qp/q.sum(1)]))
# Reproducible cost of the stationary experimental route, not a time profile.
timings=[]
for _ in range(5):
    t0=time.perf_counter();candidate=StationaryModes.build(water,128);timings.append(time.perf_counter()-t0)
t0=time.perf_counter();candidate.scalar_current(np.linspace(11,400,864));query=time.perf_counter()-t0
print('eigendecomposition median [s]:',np.median(timings),'864 radii query [s]:',query)

## 8.4. Граница результата

Здесь $N$ обрывает всё $P_N$-представление: это **не** степень $L$ в методе
с точным свободным хвостом. Проверка $N\to2N$ обязательна, особенно близко
к источнику и при выделении малого остатка $\ge2$ вычитанием.

Код пока возвращает стационарные $Q,F$. Он не даёт проверенный временной
профиль произвольного трека или направленного конечного ОМ. Следующая задача —
комплексные частоты, ветви затухающих мод и их согласование с точным ранним светом.

Спектральные разложения RTE такого класса известны; наша реализация —
независимый контроль и кандидат на backend LightHit:
Panasyuk, Schotland & Markel, *J. Phys. A* 39, 115–137 (2006),
[arXiv:math-ph/0505054](https://arxiv.org/abs/math-ph/0505054).

## Задания аспирантам

Вывести положительную сумму самостоятельно; сравнить её с Фурье–Бесселевым
маршрутом на одних параметрах. Проверить баланс радиального тока разностями.
Найти диапазон, где одной моды достаточно для заданной точности. Исследовать
смену $g$, $\mu_a$ и фазовой функции. Для будущей статьи отделить известный
асимптотический механизм от нового алгоритмического или экспериментального результата.